<a href="https://colab.research.google.com/github/yifeica0/ECS171-Group8/blob/flask/tfidf_LR_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import LabelBinarizer
from scipy.sparse import hstack

import nltk

# Download necessary NLTK data for VADER and POS tagging
# This might take a moment if not already downloaded
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng')

# Download required NLTK data (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
df = pd.read_parquet("datasets/with_stop_word/amazon_user_reviews_textANDfeature_with_sw_final.parquet")

In [ ]:
# Drop rows where 'original_text' is NaN or empty string
df.dropna(subset=['original_text'], inplace=True)
df = df[df['original_text'].str.strip() != '']

# Split data into features (X) and target (y)
X_all_features = df.drop(['sentiment'], axis=1)
y = df['sentiment']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_all_features, y, test_size=0.2, random_state=42, stratify=y)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform the 'original_text' column for training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['original_text'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['original_text'])

# Drop the 'original_text' column from X_train and X_test to concatenate with TF-IDF features
X_train_numerical = X_train.drop(columns=['original_text'])
X_test_numerical = X_test.drop(columns=['original_text'])

# Ensure 'vader_compound' is non-negative for Multinomial Naive Bayes
# If 'vader_compound' is present and has negative values, shift it to a non-negative range
if 'vader_compound' in X_train_numerical.columns:
    min_vader_compound_train = X_train_numerical['vader_compound'].min()
    if min_vader_compound_train < 0:
        X_train_numerical['vader_compound'] = X_train_numerical['vader_compound'] + abs(min_vader_compound_train)
        X_test_numerical['vader_compound'] = X_test_numerical['vader_compound'] + abs(min_vader_compound_train)

# Convert numerical features to sparse matrix format if they are not already (hstack requires sparse or numpy arrays)
X_train_numerical_sparse = X_train_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_train_numerical) else X_train_numerical.values
X_test_numerical_sparse = X_test_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_test_numerical) else X_test_numerical.values

# Combine TF-IDF features with other numerical features
X_train_combined = hstack([X_train_tfidf, X_train_numerical_sparse])
X_test_combined = hstack([X_test_tfidf, X_test_numerical_sparse])


In [ ]:
# Initialize and train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
lr_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_lr = lr_model.predict(X_test_combined)

# Evaluate the model
print("Logistic Regression Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, digits=4))